# Data Processing Pipeline
## Полный пайплайн предобработки данных

Этот нотбук реализует **10-шаговый pipeline** очистки и подготовки данных:
1. Загрузка и просмотр
2. Очистка форматирования целевого признака (Price)
3. Исправление категориальных ошибок
4. Импутация пропущенных значений
5. Удаление дубликатов
6. Ограничение выбросов (IQR capping)
7. One-hot кодирование
8. Создание признаков (feature engineering)
9. Масштабирование признаков
10. Сохранение результата

In [ ]:
!pip install -q pandas numpy scikit-learn

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

if not os.path.exists('dataset/raw_house_dataset.csv'):
    !git clone https://github.com/Enlyl/ds-ml-bootcamp.git /tmp/ds-ml-bootcamp 2>/dev/null
    os.chdir('/tmp/ds-ml-bootcamp')
    print('Repo cloned. Working directory:', os.getcwd())
else:
    print('Files found. Working directory:', os.getcwd())

## 1. Загрузка и просмотр (Load & Inspect)

In [ ]:
CSV_PATH = './dataset/raw_house_dataset.csv'
df = pd.read_csv(CSV_PATH)

print('=== INITIAL HEAD (НАЧАЛЬНЫЕ СТРОКИ) ===')
print(df.head())

print('\n=== INITIAL INFO (НАЧАЛЬНАЯ ИНФОРМАЦИЯ) ===')
print(df.info())

print('\n=== INITIAL MISSING VALUES (ПРОПУЩЕННЫЕ ЗНАЧЕНИЯ) ===')
print(df.isnull().sum())

## 2. Очистка форматирования целевого признака (Price)

In [ ]:
df['Price'] = df['Price'].replace(r'[\$,]', '', regex=True).astype(float)
print('Price dtype:', df['Price'].dtype)
print('Price skewness:', df['Price'].skew())

## 3. Исправление категориальных ошибок (до импутации)

In [ ]:
df['Location'] = df['Location'].replace({'Subrb': 'Suburb', '??': pd.NA})
print(df['Location'].value_counts(dropna=False))

## 4. Импутация пропущенных значений

In [ ]:
df['Size_sqft'] = df['Size_sqft'].fillna(df['Size_sqft'].median())
df['Bedrooms']  = df['Bedrooms'].fillna(df['Bedrooms'].mode()[0])
df['Location']  = df['Location'].fillna(df['Location'].mode()[0])

print('Missing after imputation:')
print(df.isnull().sum())

## 5. Удаление дубликатов

In [ ]:
before = df.shape
df = df.drop_duplicates()
after = df.shape
print(f'Duplicates removed: {before} → {after}')

## 6. Ограничение выбросов (IQR capping)

In [ ]:
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

low_price, high_price = iqr_fun(df['Price'])
low_size,  high_size  = iqr_fun(df['Size_sqft'])

df['Price']     = df['Price'].clip(lower=low_price, upper=high_price)
df['Size_sqft'] = df['Size_sqft'].clip(lower=low_size,  upper=high_size)

print(f'Price  range after capping: [{low_price:,.0f}, {high_price:,.0f}]')
print(f'Size   range after capping: [{low_size:,.0f}, {high_size:,.0f}]')

## 7. One-hot кодирование

In [ ]:
df = pd.get_dummies(df, columns=['Location'], drop_first=False, dtype='int')
print('New columns:', [c for c in df.columns if c.startswith('Location_')])

## 8. Создание признаков (Feature Engineering, no leakage)

In [ ]:
CURRENT_YEAR = 2025
df['HouseAge'] = CURRENT_YEAR - df['YearBuilt']
df['Rooms_per_1000sqft'] = (df['Bedrooms'] + df['Bathrooms']) / (df['Size_sqft'] / 1000)
df['Size_per_Bedroom'] = df['Size_sqft'] / df['Bedrooms'].replace(0, np.nan)
df['Is_City'] = df['Location_City'].astype(int)
df['LogPrice'] = np.log1p(df['Price'])

print('New features:', ['HouseAge', 'Rooms_per_1000sqft', 'Size_per_Bedroom', 'Is_City', 'LogPrice'])

## 9. Масштабирование признаков (только X)

In [ ]:
dont_scale = {'Price', 'LogPrice'}
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.to_list()
exclude = [c for c in df.columns if c.startswith('Location_')] + ['Is_City']
num_features_to_scale = [c for c in numeric_cols if c not in dont_scale and c not in exclude]

scaler = StandardScaler()
df[num_features_to_scale] = scaler.fit_transform(df[num_features_to_scale])

print('Scaled columns:', num_features_to_scale)
print('\nSample of scaled values:')
print(df[num_features_to_scale].head())

## 10. Финальные проверки и сохранение

In [ ]:
print('=== FINAL HEAD (ИТОГОВЫЕ СТРОКИ) ===')
print(df.head())

print('\n=== FINAL INFO (ИТОГОВАЯ ИНФОРМАЦИЯ) ===')
print(df.info())

print('\n=== FINAL MISSING VALUES (ПРОПУЩЕННЫЕ ЗНАЧЕНИЯ) ===')
print(df.isnull().sum())

In [ ]:
OUT_PATH = './dataset/clean_house_dataset.csv'
df.to_csv(OUT_PATH, index=False)
print(f'Saved to {OUT_PATH} — shape: {df.shape}')